In [ ]:
# --- 0. Imports and Setup ---
import scanpy as sc
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# Adjust the path to import your package
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from scpred_py_final import ScPredModel
from scpred_py_final import _analysis_utils

# --- Environment Setup ---
print("--- Setting up environment for reproducibility ---")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
sc.settings.set_figure_params(dpi=100, facecolor='white', figsize=(8, 8))

# --- 1. Data Loading and Annotation ---
print("\n--- Step 1: Loading and Annotating paul15 Data ---")
adata = sc.datasets.paul15()
adata.var_names_make_unique()
adata.obs['cell_type'] = adata.obs['paul15_clusters'].astype('category')
print("Data loading and labeling complete.")

# --- 2. Train-Test Split ---
print("\n--- Step 2: Splitting Data ---")
indices = range(adata.n_obs)
ref_idx, query_idx = train_test_split(
    indices, test_size=0.3, random_state=RANDOM_STATE, stratify=adata.obs['cell_type']
)
ref_adata_raw = adata[ref_idx, :].copy()
query_adata_raw = adata[query_idx, :].copy()
print("Data splitting complete.")

# --- 3. Training the Optimal scPred Model for paul15 ---
print("\n--- Step 3: Training Optimal Model (RBF SVM, Balanced) ---")
scpred_model = ScPredModel()

# Train using the best parameters
scpred_model.train(
    ref_adata=ref_adata_raw,
    cell_type_key='cell_type',
    n_components=30,
    hvg_n_top_genes=2000,
    svm_kernel='rbf',
    svm_c=1.0,
    svm_random_state=RANDOM_STATE,
    svm_class_weight='balanced'
)
print("\nOptimal model trained!")

# --- 4. Prediction and Evaluation at Different Thresholds ---
# (This helper function is identical to the pbmc3k one)
def run_prediction_and_eval(model, query_data, threshold):
    """Helper function to run prediction and print a summary."""
    print("\n" + "="*50)
    print(f"  RUNNING PREDICTION WITH THRESHOLD: {threshold}")
    print("="*50)

    query_pred = model.predict(query_data.copy(), threshold=threshold)
    print("\nPredicted label distribution:")
    print(query_pred.obs['scpred_prediction'].value_counts(dropna=False))
    
    prob_cols = [col for col in query_pred.obs.columns if col.startswith('scpred_prob_')]
    y_pred_probs_df = query_pred.obs[prob_cols].copy() if prob_cols else None

    _analysis_utils.evaluate_and_report_metrics(
        true_labels=query_pred.obs['cell_type'],
        predicted_labels=query_pred.obs['scpred_prediction'],
        classifier_classes=model.classifier_.classes_,
        y_pred_probs=y_pred_probs_df
    )
    return query_pred

# Run with threshold=0.0 (full coverage)
query_pred_t0 = run_prediction_and_eval(scpred_model, query_adata_raw, threshold=0.0)

# Run with threshold=0.8 (high confidence)
query_pred_t8 = run_prediction_and_eval(scpred_model, query_adata_raw, threshold=0.8)

# --- 5. Final Visualizations for the Report ---
print("\n--- Step 5: Final UMAP Visualizations ---")
sc.pp.neighbors(query_pred_t8, n_neighbors=10, use_rep='X_scpred_pca', random_state=RANDOM_STATE)
sc.tl.umap(query_pred_t8, random_state=RANDOM_STATE)

print("\n--- UMAP: True Labels vs. High-Confidence Predictions (Threshold=0.8) ---")
sc.pl.umap(
    query_pred_t8,
    color=['cell_type', 'scpred_prediction'],
    title=['True Labels', f'scPred Predictions (Threshold {0.8})'],
    frameon=False, wspace=0.4, legend_loc='on data'
)

# Plotting the confidence of the model
print("\n--- UMAP: Prediction Confidence ---")
query_pred_t8.obs['max_probability'] = query_pred_t8.obs[[c for c in query_pred_t8.obs.columns if c.startswith('scpred_prob_')]].max(axis=1)
sc.pl.umap(
    query_pred_t8,
    color='max_probability',
    cmap='viridis',
    title='Prediction Confidence on Assigned Cells',
    frameon=False
)

print("\n--- Analysis Complete ---")